# 04b — Feature Engineering

**Goal (game plan Week 6):** engineer new features, add a school-district layer by spatially joining each home to the
[CA School District Areas 2024-25](https://data.ca.gov/dataset/california-school-district-areas-2024-25) boundaries,
retrain the models, and compare the old and new feature sets.
**Author:** Ethan Chung — IDX Exchange Data Science, Fall 2026

**New feature families** (`src/features.py`, `src/geo.py`):

| Family | Features | Notes |
|---|---|---|
| Ratios | `bed_bath_ratio`, `sqft_per_bedroom` | suggested in the game plan; age at sale (`property_age`) already existed |
| Flooring | one 0/1 column per flooring type + missing flag | vocabulary learned on training data only (types with ≥ 500 training homes) |
| School districts | K-8 district, high-school district (target-encoded), district type, K-8 enrollment, state assistance status | spatial join; 99.1% of homes matched |

**Fair-housing note:** the district layer also carries race/ethnicity, English-learner and low-income shares. Those are **not used**: a valuation model that learns from neighborhood demographics can discount homes by who lives nearby, the risk AVM fair-lending rules target.

**How the comparison works:** every model keeps the training window and hyperparameters it chose in Week 5, so only the feature set changes. Which features to adopt is decided on the **2026-03 validation month**; test (2026-04) is reported alongside.

Run first: `python src/geo.py`, the comparison (`models.compare_feature_sets()`), and `python src/models.py --features week6`.

## 1. Setup

In [ ]:
import os
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import evaluation
import features
import models
import preprocessing

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
LABELS = {"baseline_linear_regression": "Linear Regression", "decision_tree": "Decision Tree", "random_forest": "Random Forest"}

clean = preprocessing.load_clean("../data/processed/sfr_clean.parquet")
train, val, test, info = preprocessing.make_split(clean, 24)
X_train, y_train = features.make_xy(train)

## 2. The school-district layer

In [ ]:
districts = pd.read_parquet("../data/processed/school_districts.parquet")
print(f"Matched to a district: {districts['school_district_k8'].notna().mean():.1%} of {len(districts):,} homes")
print(f"K-8 districts: {districts['school_district_k8'].nunique()}, high-school districts: {districts['school_district_hs'].nunique()}")
districts["district_type"].value_counts(dropna=False).to_frame("homes")

In [ ]:
# The MLS field HighSchoolDistrict is blank for about a quarter of homes; the spatial join fills almost all of them
m = clean[["ListingKey", "HighSchoolDistrict"]].merge(districts, on="ListingKey")
blank = m["HighSchoolDistrict"].isna()
print(f"MLS HighSchoolDistrict blank: {blank.mean():.1%} of homes; of those, the join assigns a district to {m.loc[blank, 'school_district_hs'].notna().mean():.1%}")
districts["district_k8_assistance"].value_counts(dropna=False).to_frame("homes")

## 3. Flooring types and ratios

In [ ]:
flooring = features.MultiHot().fit(X_train[["Flooring"]])
flags = pd.DataFrame(flooring.transform(X_train[["Flooring"]]), columns=flooring.get_feature_names_out())
(flags.mean() * 100).sort_values(ascending=False).to_frame("% of training homes")

In [ ]:
X_train[["bed_bath_ratio", "sqft_per_bedroom"]].describe(percentiles=[0.01, 0.5, 0.99]).T

## 4. Old vs new feature sets

In [ ]:
comp = pd.read_csv("../docs/feature_set_comparison.csv")
comp["model"] = comp["model"].map(LABELS)
order = ["week5", "week5+ratios", "week5+flooring", "week5+school_districts", "week6", "week6+ratios"]
table = comp.pivot_table(index=["model", "feature_set"], columns="stage", values=["MdAPE", "R2"], sort=False)
base = table.xs("week5", level="feature_set")
delta = (table["MdAPE"]["validation"] - base["MdAPE"]["validation"].reindex(table.index.get_level_values(0)).to_numpy())
table[("MdAPE", "validation change vs week5")] = delta.to_numpy()
table.round(3)

In [ ]:
val_mdape = comp[comp["stage"] == "validation"].pivot(index="feature_set", columns="model", values="MdAPE").loc[order]
change = val_mdape - val_mdape.loc["week5"]
ax = change.drop(index="week5").plot(kind="bar", figsize=(9, 3.8), rot=20,
                                     title="Validation MdAPE change vs Week 5 features (points; negative = better)")
ax.axhline(0, color="black", linewidth=0.8); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Is the Random Forest gain more than noise?

Paired bootstrap on the validation month: same homes, Week 5 features vs the chosen `week6` set (same window and hyperparameters). Takes a few minutes: two Random Forest fits.

In [ ]:
rf_factory, _ = models.MODELS["random_forest"]
rf_params = {"max_features": 0.33, "min_samples_leaf": 1}   # chosen on validation in Week 5
pred = {fs: models.predict_dollars(models.fit(models.make_model(rf_factory(**rf_params), fs), train), val)
        for fs in ["week5", "week6"]}
pd.Series(evaluation.paired_bootstrap_mdape(val["ClosePrice"], pred["week5"], pred["week6"]))

**Decision (on validation):** ratios make both tree models worse, so the final `week6` set is **Week 5 + flooring + school districts**.
It is the best set for both tree models; Linear Regression would gain slightly more with ratios included, but it is the baseline, not the production candidate.

## 5. Final models retrained on `week6` (full window + hyperparameter selection)

In [ ]:
rows = []
for n in LABELS:
    for fs in ["week5", "week6"]:
        m = pd.read_csv(f"../docs/metrics_{models.artifact_stem(n, fs)}.csv")
        t = m[m["stage"] == "test"].iloc[0]
        bt = m[m["stage"] == "backtest"]["MdAPE"]
        rows.append({"model": LABELS[n], "features": fs, "window": t["train_months"], "params": t["params"],
                     "test R2": t["R2"], "test MdAPE": t["MdAPE"], "test MAPE": t["MAPE"], "test MAE": t["MAE"],
                     "within 10%": t["within_10pct"], "backtest MdAPE": f"{bt.min():.2f}–{bt.max():.2f}"})
final = pd.DataFrame(rows)
final

In [ ]:
rf = joblib.load(f"../models/{models.artifact_stem('random_forest', 'week6')}.joblib")["model"]
imp = pd.Series(rf.named_steps["model"].feature_importances_, index=rf.named_steps["preprocess"].get_feature_names_out())
print(f"School-district features: {imp.filter(like='district').sum():.1%} of importance, flooring: {imp.filter(like='Flooring').sum():.1%}")
imp.sort_values(ascending=False).head(15).to_frame("importance (impurity decrease)")

## 6. Findings

**Old vs new feature set, final models (test 2026-04)**

| Model | Week 5 features: MdAPE / R² | Week 6 features: MdAPE / R² | Backtest MdAPE, week5 → week6 |
|---|---|---|---|
| Linear Regression | 11.23% / 0.822 | **10.99% / 0.827** | 11.13–11.36% → 10.82–11.24% |
| Decision Tree | 9.30% / 0.858 | 9.32% / 0.845 | 9.14–9.50% → 9.03–9.49% |
| Random Forest | 7.60% / 0.896 | **7.52% / 0.897** | 7.50–7.79% → 7.39–7.68% |

- **The school-district layer is the most useful addition.** It improves every model on validation, fills the MLS district field for the 26% of homes where it was blank, and the K-8 district becomes the Random Forest's 4th most important feature (districts ≈ 15% of total importance).
- **Flooring helps a little** on validation (≈ 0.5% of Random Forest importance).
- **Ratios (bed/bath, sq ft per bedroom) hurt both tree models** on validation, so they are not in the final set: trees already combine beds, baths and size themselves, and the ratios add noise. They do help Linear Regression slightly.
- **Random Forest gains are small but consistent:** MdAPE is 0.07–0.11 points lower in every one of the four test months. On the validation month alone the gain is borderline (+0.10 points, 95% interval −0.002 to +0.22), so the honest summary is "a small, steady improvement".
- **Decision Tree:** re-tuning picked a shallower tree (depth 16, leaf 5) that won on validation but is flat on test, a reminder that single trees are noisy.
- **Linear Regression gains the most** (−0.24 points on test), because it cannot build location interactions on its own.

**Assumptions and limitations**
- District boundaries and enrollment are the 2024-25 snapshot, applied to sales from 2024–2026.
- 1% of homes have no coordinates or fall outside every district; they get a "missing" district and an imputed enrollment.
- District demographic shares are deliberately excluded (fair-housing risk), even though they might add accuracy.

**Next (Week 7):** gradient boosting (XGBoost / LightGBM) on the `week6` feature set with light tuning on the validation month.